In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
# Import neccesary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from tqdm import tqdm
import math

%matplotlib inline

csv_path = os.path.join(path, "Q1_data.csv")

df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
# Maybe used as a utiliy function later
def check_target_distribution(df, target_column):
  # Plot a histogram of the ditribution
  df[target_column].hist(bins=30, edgecolor='black')

  # Titles
  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()
target_column = 'Delivery_Time'
check_target_distribution(df, target_column)

In [ ]:
# Task 1: Write your code here:
# Do not use the 'inplace=True' parameter as it requires reload the data for changes to be undone
# axis=1 as this is a column
df = df.drop('Order_ID', axis=1)

In [ ]:
# Task 2: Write your code here:
# 2. Do we have missing values?
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])

check_missing_values(df)

missing_cols = ['Weather',
'Traffic_Level',
'Time_of_Day',
'Courier_Experience_yrs',
'Delivery_Time']
df[missing_cols]
# We will save the missing cols for now. We will fill them later when we have encoded categorical columns (task 4)

In [ ]:
# Task 3: Write your code here:
# Just drop them. We don't need them
df = df.drop_duplicates()

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import LabelEncoder
# let's check them
categorical_cols = df.select_dtypes(include=["object"]).columns
print("Categorical Columns:", list(categorical_cols))

# Label encoding
label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col])
  label_encoders[col] = le


# Let's remember to fill missing values with the mean now that we have encoded the values
for col in missing_cols:
  df = df.fillna(df[col].mode()[0])

df

In [ ]:
# Task 5: Write your code here:

from sklearn.preprocessing import StandardScaler
# 'number' is more general than float64 and int64 specifically
numerical_cols = df.select_dtypes(include=['number']).columns.drop(target_column)  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df


In [ ]:
# Task 6: Write your code here:

In [ ]:
# Task 1: Write your code here:
X = df.drop(target_column, axis=1).astype(float)
y = df[target_column].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
# We might tune this later
n_splits = 5  # K=5 Folds

# 5-Fold Cross-Validation, shuffled
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
# Loss metric
all_mae = []
# Training model, might tune n_estimators later too
model = RandomForestRegressor(n_estimators=200)
for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]


  # Train
  model.fit(X_train, y_train)

  # Predict
  y_pred = model.predict(X_test)

  # Calculate mae
  mae = mean_absolute_error(y_test, y_pred)
  all_mae.append(mae)

print(f"MAE:  {np.mean(all_mae):.4f}")

In [ ]:
# Let's check the baseline
# Calculate the baseline predictions (mean of the target)
baseline_pred = np.full_like(y, y.mean())

# Evaluate the baseline
baseline_mae = mean_absolute_error(y, baseline_pred)

print(f"Baseline MAE (using mean target): {baseline_mae:.4f}")
# Our model is better than the basline

In [ ]:
# Task 1: Write your code here:
imp = model.feature_importances_
abs_imp = np.abs(imp)
sorted_idx = np.argsort(abs_imp)
features = X.columns

# Create a plot
plt.figure(figsize=(10,6))
plt.barh(features[sorted_idx], abs_imp[sorted_idx])
plt.title('Random Forest Regressor Feature Importances')
plt.show()

In [ ]:
# Task 2: Write your code here:
train_index, test_index = next(iter(kf.split(X)))
X_train, X_test = X.iloc[train_index], X.iloc[test_index]
y_train, y_test = y.iloc[train_index], y.iloc[test_index]
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
axes = axes.flatten()
axes[0].hist(y)
axes[0].set_title('Actual Value')
axes[1].hist(y_pred)
axes[1].set_title('Predicted Value')


In [ ]:
# install catboost
%pip install catboost

In [ ]:
# Task Bonus: Write your code here:
from catboost import CatBoostRegressor
sklearn_models = {
  "Random Forest": RandomForestRegressor(
      n_estimators=320,  # Number of trees
      max_depth=4
  ),
  "CatBoost": CatBoostRegressor(
      verbose=0,
      n_estimators=320,
      max_depth=4
  )
}
all_results = {}

for name in sklearn_models:
  all_results[name] = {'MAE': []}
n_splits = 5 # K

# Stratified 5-Fold Cross-Validation, shuffled
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # 2. Train & Validate sklearn models
  for model_name, model in sklearn_models.items():
    print(f"Training {model_name}...")
    model.fit(X_train, y_train) # train
    y_pred = model.predict(X_test) # validate

    # 3. Save metrics for that model in this fold
    mae = mean_absolute_error(y_test, y_pred)

    all_results[model_name]['MAE'].append(mae)
cat_avg = np.mean(all_results['CatBoost']['MAE'])
rf_avg = np.mean(all_results['Random Forest']['MAE'])
print(f"Average MAE:  {(cat_avg + rf_avg) / 2:.4f}")